# Úkol 3c: Structured Outputs nad DataCorp daty

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/shippy/czechitas-ai-data/blob/main/notebooks/assignment-03c.ipynb)

Pokud nebudete vědět, nezapomeňte se zeptat [pomocníčka pro tento úkol](https://chatgpt.com/g/g-67cab661ae5c8191b0d8419c76d3959b-czechitas-ai-in-data-analytics-2025-03)!

## Kontext

V tomto notebooku použijeme structured outputs k analýze nestrukturovaných textových dat z DataCorpu:
- **Performance reviews** (`datacorp_reviews.csv`) — hodnocení zaměstnanců jejich nadřízenými
- **Exit interviews** (`datacorp_exit_interviews.csv`) — rozhovory s odcházejícími zaměstnanci

Cílem je vytáhnout z volného textu strukturované informace, které pak můžeme propojit s hlavním datasetem `datacorp.csv`.

## Setup

Spusťte následující dvě buňky. Pokud vám AI řekne, jak se má, vše funguje správně.

In [ ]:
try:
    from google.colab import userdata
    _secret = userdata.get("OPENAI_API_KEY")
    %pip install pandas instructor openai openpyxl python-dotenv rich seaborn
    !curl -s https://raw.githubusercontent.com/shippy/czechitas-ai-data/refs/heads/main/notebooks/datacorp.csv > datacorp.csv
    !curl -s https://raw.githubusercontent.com/shippy/czechitas-ai-data/refs/heads/main/notebooks/datacorp_reviews.csv > datacorp_reviews.csv
    !curl -s https://raw.githubusercontent.com/shippy/czechitas-ai-data/refs/heads/main/notebooks/datacorp_exit_interviews.csv > datacorp_exit_interviews.csv
    !curl -s https://raw.githubusercontent.com/shippy/czechitas-ai-data/refs/heads/main/notebooks/datacorp_payroll_q3.xlsx > datacorp_payroll_q3.xlsx
except ImportError:
    import os
    from dotenv import load_dotenv
    _ = load_dotenv()
    _secret = os.environ.get("OPENAI_API_KEY")


In [ ]:
import asyncio
import instructor
import os
from rich import print

try:
    if not _secret:
        raise ValueError("API klíč nebyl nastaven!")
except NameError:
    print("Nastavte si API klíč v proměnné OPENAI_API_KEY a znovu spusťte *celý* notebook včetně předchozí buňky")

os.environ["OPENAI_API_KEY"] = _secret

# Synchronní klient — pro test a jednoduché volání
client = instructor.from_provider("openai/gpt-5.4-mini")

# Asynchronní klient — pro paralelní zpracování mnoha řádků
async_client = instructor.from_provider("openai/gpt-5.4-mini", async_client=True)

# Semaphore omezuje počet souběžných požadavků (abychom nepřetížili API)
SEM = asyncio.Semaphore(5)

test = client.create(
    messages=[
        {"role": "user", "content": "Hello, how are you? Respond with an emotion."},
    ],
    response_model=str,
)
print(test)

## Čtení dat

In [ ]:
import pandas as pd

datacorp = pd.read_csv("datacorp.csv")
reviews = pd.read_csv("datacorp_reviews.csv")
exit_interviews = pd.read_csv("datacorp_exit_interviews.csv")

print(f"Zaměstnanci: {datacorp.shape[0]} řádků, Reviews: {reviews.shape[0]} řádků, Exit interviews: {exit_interviews.shape[0]} řádků")

In [ ]:
reviews.head()

In [ ]:
exit_interviews.head()

## Ukázka: Strukturovaný výstup z exit interview

Podívejme se, jak z volného textu exit interview vytáhnout strukturované informace:

In [ ]:
from pydantic import BaseModel, Field
from typing import Literal

class ExitReason(BaseModel):
    primary_reason: Literal["plat", "vedení", "kariérní růst", "kultura", "jiná nabídka", "osobní důvody", "jiné"]
    sentiment: Literal["pozitivní", "neutrální", "negativní"]
    summary: str = Field(..., description="Shrnutí v jedné větě")

async def extract_exit_reason(text: str) -> ExitReason:
    async with SEM:
        return await async_client.create(
            messages=[
                {"role": "system", "content": "Analyzuj text z exit interview zaměstnance. Extrahuj hlavní důvod odchodu, sentiment a shrnutí."},
                {"role": "user", "content": text},
            ],
            response_model=ExitReason,
        )

# Ukázka na prvním záznamu
example = exit_interviews.iloc[0]
print(f"Text: {example['interview_text']}\n")

result = await extract_exit_reason(example["interview_text"])
print(result)

## Úkol 1: Analýza exit interviews

Zpracujte **všechny** exit interviews pomocí structured outputs. Pro každý záznam extrahujte:
- hlavní důvod odchodu
- sentiment (pozitivní / neutrální / negativní)
- jednověté shrnutí

Výsledky uložte do nového sloupce `exit_interviews['structured']`.

In [ ]:
results = await asyncio.gather(*[
    extract_exit_reason(row["interview_text"])
    for _, row in exit_interviews.iterrows()
])

exit_interviews["structured"] = results
print(f"Zpracováno {len(results)} exit interviews")
exit_interviews[["employee_id", "structured"]].head()

## Úkol 2: Analýza performance reviews

Nadefinujte vlastní Pydantic model pro performance reviews a zpracujte všechny záznamy. Zamyslete se, jaké informace lze z review textu vytáhnout — např.:
- celkový sentiment hodnocení
- silné stránky zaměstnance
- oblasti ke zlepšení
- doporučená akce (povýšení, rozvojový plán, varování, …)

Tip: podívejte se na několik review textů a na základě nich navrhněte strukturu.

In [ ]:
class ReviewAnalysis(BaseModel):
    sentiment: Literal["pozitivní", "neutrální", "negativní"]
    strengths: list[str] = Field(default_factory=list, description="Silné stránky zaměstnance zmíněné v review")
    areas_to_improve: list[str] = Field(default_factory=list, description="Oblasti ke zlepšení zmíněné v review")
    recommended_action: Literal[
        "navýšení platu", "povýšení", "rozvojový plán",
        "školení", "přesun do jiného týmu", "varování", "žádná"
    ] = Field(..., description="Doporučená akce na základě review")
    summary: str = Field(..., description="Shrnutí review v jedné větě")

async def extract_review(text: str) -> ReviewAnalysis:
    async with SEM:
        return await async_client.create(
            messages=[
                {"role": "system", "content": "Analyzuj text z performance review zaměstnance. Extrahuj sentiment, silné stránky, oblasti ke zlepšení, doporučenou akci a shrnutí."},
                {"role": "user", "content": text},
            ],
            response_model=ReviewAnalysis,
        )

review_results = await asyncio.gather(*[
    extract_review(row["review_text"])
    for _, row in reviews.iterrows()
])

reviews["structured"] = review_results
print(f"Zpracováno {len(review_results)} reviews")
reviews[["employee_id", "rok", "structured"]].head()

## Úkol 3: Propojení s hlavním datasetem

Propojte výsledky z úkolů 1 a 2 s hlavním datasetem `datacorp.csv` přes sloupec `employee_id`. Pak zodpovězte:

1. **Které oddělení má nejvíc negativních exit interviews?**
2. **Koreluje sentiment review s výší platu nebo hodnocením výkonu?**
3. **Jaké jsou nejčastější důvody odchodu?** Vytvořte graf.

Tip: Vytvořte si pomocné sloupce z extrahovaných dat (např. `exit_interviews['primary_reason'] = [r.primary_reason for r in results]`).

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

# Extrahujeme pole z Pydantic modelů do sloupců
exit_interviews["primary_reason"] = [r.primary_reason for r in results]
exit_interviews["sentiment"] = [r.sentiment for r in results]

reviews["review_sentiment"] = [r.sentiment for r in review_results]
reviews["recommended_action"] = [r.recommended_action for r in review_results]

# Propojení s hlavním datasetem
exit_merged = exit_interviews.merge(datacorp, on="employee_id", how="left")
reviews_merged = reviews.merge(datacorp, on="employee_id", how="left")

In [ ]:
# 3.1: Které oddělení má nejvíc negativních exit interviews?
neg_by_dept = (
    exit_merged[exit_merged["sentiment"] == "negativní"]
    .groupby("oddeleni")
    .size()
    .sort_values(ascending=False)
)
print("Negativní exit interviews podle oddělení:")
print(neg_by_dept)

fig, ax = plt.subplots(figsize=(8, 4))
neg_by_dept.plot.bar(ax=ax, color="#E6007E")
ax.set_title("Negativní exit interviews podle oddělení")
ax.set_ylabel("Počet")
ax.set_xlabel("")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

In [ ]:
# 3.2: Koreluje sentiment review s výší platu nebo hodnocením výkonu?
sentiment_map = {"pozitivní": 1, "neutrální": 0, "negativní": -1}
reviews_merged["sentiment_score"] = reviews_merged["review_sentiment"].map(sentiment_map)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

sns.boxplot(data=reviews_merged, x="review_sentiment", y="plat",
            order=["negativní", "neutrální", "pozitivní"], ax=axes[0])
axes[0].set_title("Plat vs. sentiment review")
axes[0].set_xlabel("Sentiment")
axes[0].set_ylabel("Plat (Kč)")

sns.boxplot(data=reviews_merged, x="review_sentiment", y="hodnoceni_vykonu",
            order=["negativní", "neutrální", "pozitivní"], ax=axes[1])
axes[1].set_title("Hodnocení výkonu vs. sentiment review")
axes[1].set_xlabel("Sentiment")
axes[1].set_ylabel("Hodnocení výkonu")

plt.tight_layout()
plt.show()

print(f"\nKorelace sentiment_score vs. plat: {reviews_merged['sentiment_score'].corr(reviews_merged['plat']):.3f}")
print(f"Korelace sentiment_score vs. hodnocení: {reviews_merged['sentiment_score'].corr(reviews_merged['hodnoceni_vykonu']):.3f}")

In [ ]:
# 3.3: Jaké jsou nejčastější důvody odchodu?
reason_counts = exit_interviews["primary_reason"].value_counts()

fig, ax = plt.subplots(figsize=(8, 4))
reason_counts.plot.barh(ax=ax, color="#2D2E83")
ax.set_title("Nejčastější důvody odchodu (exit interviews)")
ax.set_xlabel("Počet")
ax.set_ylabel("")
ax.invert_yaxis()
plt.tight_layout()
plt.show()

## Úkol 4: Rekonciliace mzdového listu (xlsx)

Finance vám poslalo soubor `datacorp_payroll_q3.xlsx` — kvartální mzdový list za Q3 2025. Některé řádky ale **nesedí** s tím, co máme v `datacorp.csv`: jiné oddělení, jiná částka, nebo dokonce hodnota v jiné měně. Vaším úkolem je použít structured outputs jako auditora a nechat LLM rozhodnout, co je v pořádku a co ne.

### 4a. Načtěte mzdový list

Soubor je formát `.xlsx`, ne CSV. Pozor:

- Horní 3 řádky jsou hlavičkové artefakty (z původní šablony).
- Poslední řádek je součtový (`CELKEM`), který chcete odfiltrovat.
- Pracujte s listem `Mzdy Q3 2025` (workbook obsahuje i prázdné `List2` a `List3`).

Použijte `pd.read_excel` se vhodným argumentem `skiprows=` a odfiltrujte součtový řádek.

In [ ]:
payroll = pd.read_excel(
    "datacorp_payroll_q3.xlsx",
    sheet_name="Mzdy Q3 2025",
    skiprows=3,
)
payroll = payroll[payroll["os_cislo"].apply(lambda x: str(x).replace(".0", "").isdigit())]
payroll["os_cislo"] = payroll["os_cislo"].astype(int)
print(f"Payroll rows after cleanup: {len(payroll)}")
payroll.head()


In [ ]:
# Vzorek 50 řádků s pevně daným seedem.
# Seed je záměrně zvolen tak, aby vzorek obsahoval alespoň jeden řádek v EUR
# a alespoň jeden řádek se špatným oddělením — ať vidíme, co LLM zachytí.

payroll_sample = payroll.sample(50, random_state=5).reset_index(drop=True)
print(f"Vzorek: {len(payroll_sample)} řádků")
payroll_sample.head()

In [ ]:
from enum import Enum
from pydantic import BaseModel, Field


class Zavaznost(str, Enum):
    OK = "ok"
    DROBNA = "drobná"
    STREDNI = "střední"
    VAZNA = "vážná"


class Nesoulad(BaseModel):
    pole: str = Field(description="Název pole, ve kterém je nesoulad (např. 'oddeleni', 'mzda_brutto').")
    hodnota_hr: str
    hodnota_payroll: str
    pravdepodobna_pricina: str = Field(description="Stručné vysvětlení, proč nesoulad nejspíš vznikl.")


class Verdikt(BaseModel):
    stejna_osoba: bool = Field(description="Týkají se obě řádky téhož zaměstnance?")
    nesoulady: list[Nesoulad]
    zavaznost: Zavaznost
    doporuceni: str = Field(description="Jedna věta — co s tím udělat.")

In [ ]:
async def reconcile_row(payroll_row: pd.Series, hr_row: pd.Series) -> Verdikt:
    user_block = (
        f"HR záznam (zdroj: datacorp.csv, plat v CZK):\n"
        f"  employee_id={hr_row['employee_id']} jmeno={hr_row['jmeno']} prijmeni={hr_row['prijmeni']} "
        f"oddeleni={hr_row['oddeleni']} plat={hr_row['plat']}\n\n"
        f"Mzdový záznam (zdroj: datacorp_payroll_q3.xlsx):\n"
        f"  os_cislo={payroll_row['os_cislo']} jmeno_prijmeni={payroll_row['jmeno_prijmeni']} "
        f"oddeleni={payroll_row['oddeleni']} mzda_brutto={payroll_row['mzda_brutto']} "
        f"bonus={payroll_row['bonus']} mzda_celkem={payroll_row['mzda_celkem']} "
        f"aktivni={payroll_row['aktivni']}"
    )
    async with SEM:
        return await async_client.create(
            messages=[
                {"role": "system", "content": (
                    "Jsi pomocný auditor. Porovnáš dva záznamy o jednom zaměstnanci ze dvou systémů "
                    "(HR a mzdové oddělení) a vrátíš strukturovaný verdikt. Mzdy v CZK obvykle spadají "
                    "do rozsahu 25 000 – 150 000. Pokud vidíš číslo mimo tento rozsah, zvaž, jestli "
                    "není v jiné měně."
                )},
                {"role": "user", "content": user_block},
            ],
            response_model=Verdikt,
        )


# Spojíme vzorek payrollu s HR daty přes employee_id / os_cislo
merged = payroll_sample.merge(
    datacorp, left_on="os_cislo", right_on="employee_id", how="left",
    suffixes=("_payroll", ""),
)
verdicts = await asyncio.gather(*[
    reconcile_row(payroll_sample.loc[i], merged.loc[i])
    for i in payroll_sample.index
])
payroll_sample["verdikt"] = verdicts
print(f"Zpracováno {len(verdicts)} řádků")

In [ ]:
# 1. Distribuce závažnosti
zavaznosti = payroll_sample["verdikt"].apply(lambda v: v.zavaznost.value)
print(zavaznosti.value_counts())

# 2. Vážné případy
vazne = payroll_sample[zavaznosti == "vážná"]
for _, r in vazne.iterrows():
    print(f"os_cislo={r['os_cislo']}  oddeleni(payroll)={r['oddeleni']}  mzda_brutto={r['mzda_brutto']}")
    for n in r["verdikt"].nesoulady:
        print(f"  - {n.pole}: HR={n.hodnota_hr!r} vs Payroll={n.hodnota_payroll!r} ({n.pravdepodobna_pricina})")

# 3. Ground-truth check — full dataset má 4 EUR + 3 wrong-dept rows
hr_dept = dict(zip(datacorp["employee_id"], datacorp["oddeleni"]))
cs_depts = {"Vývoj", "Marketing", "Obchod", "Podpora", "Finance", "HR"}
eur_in_sample = ((payroll_sample["mzda_brutto"] >= 1500) & (payroll_sample["mzda_brutto"] <= 4000)).sum()
wrong_in_sample = sum(
    1 for _, r in payroll_sample.iterrows()
    if hr_dept.get(r["os_cislo"]) is not None
    and r["oddeleni"] != hr_dept[r["os_cislo"]]
    and r["oddeleni"] in cs_depts
)
print(f"Ve vzorku: {eur_in_sample} EUR-řádků, {wrong_in_sample} wrong-dept řádků.")
print(f"LLM označil jako 'vážná': {(zavaznosti == 'vážná').sum()}")


### Diskusní otázky (k zamyšlení)

1. Které typy chyb LLM odhalí spolehlivě, a které ne? Proč?
2. Co by se stalo, kdyby system prompt neobsahoval nápovědu o rozsahu CZK? Vyzkoušejte.
3. Jak by se dala metoda škálovat na celý dataset (1000 řádků) bez zbytečného plýtvání tokeny? (Tip: batch, pre-filtering, levnější model na první pass.)

## Bonusový úkol: Vylepšete svůj model

- Zkuste přidat do `ExitReason` nebo `ReviewAnalysis` další pole — např. `konkrétní_stížnosti: list[str]` nebo `doporučení_pro_firmu: str`.
- Zkuste změnit system prompt a porovnejte, jak se změní výsledky.
- Zkuste použít `Field(description=...)` pro přesnější výstupy.

In [ ]:
# Zde řešte bonusový úkol.